Note: This review is not comprehensive. Make sure you take notes in class on other things to review for the midterm.

In [18]:
import pandas as pd
import sqlite3

For this review, we will use a database with two relations in it:  

<span style="color:pink">Relation 1: Category(Incident_Type, Category)</span>
- with <span style="color:pink"> Incident_Type </span> as the Primary Key  
  
<span style="color:lavender">Relation 2: Incidents_Money(Incident_ID, Incident_Type, Estimated_Contents_Loss,Civilian_Injuries)</span>
- with <span style="color:lavender"> Incident_ID</span> as the Primary Key
- and <span style="color:lavender">Incident_Type</span> as a Foreign Key  

<span style="color:silver">Incident_ID cannot be null in the Incidents_Money table</span> 
and <span style="color:silver">Incident_Type cannot be null in the Category table.</span>

**Extra Info**
This data was sourced from publicly available information from the Virginia Beach Fire Department. Categories were made up for the purpose of this exercise and are not part of VBFD data.  

---
<span style="color:lightcyan">Q1 - Given that both primary keys are valid, why wouldn't we just put all of these columns into the same table? What normalization properties would that violate?</span>

Putting everything into one table would produce the dependency chain:

<span style="color:lightcyan">Incident_ID -> Incident Type -> Category</span>

which would violate the 3NF requirement of no transitive dependencies. Category in the Category relation is uniquely determined by Identity Type. And Identity Type in the Incidents_Money table is uniquely determined by Incidents_ID. The separate tables prevents this transitive dependency from occuring. 
    

---
<span style="color:lightcyan">Q2 - If Incident_ID is a valid primary key for the Incidents_Money table, what attributes must be in its closure?</span>





By definition, a primary key determines every attribute in its relation.

<span style="color:lightcyan">{Incident_ID}+ = {Incident_ID, Incident_Type, Estimated_Contents_Loss, Civilian_Injuries}</span>

Every attribute, including itself, must be in the closure/span

### Now we will start working with the database

In [19]:
conn = sqlite3.connect("fire.db")
# conn is connection
# opens database conection

In [20]:
curs = conn.cursor()
# curs is cursor
# executes SQL -- database object that applies queries to each row

In [21]:
curs.execute("PRAGMA foreign_keys=ON;")
# PRAGMA are adminsitrative tools to inspect the database's internal structure etc
# engables foreign-key enforcement for that connection

In [31]:
sql = """
PRAGMA table_list;
"""
# assigning text to sql does not execute it
# each new assignment replaces the previous query
# successful writes need conn.commit() to persist

In [32]:
pd.read_sql(sql, conn)
# lists tables

,schema,name,type,ncol,wr,strict
0,main,incidents_money,table,4,0,0
1,main,Category,table,2,0,0
2,main,sqlite_schema,table,5,0,0
3,temp,sqlite_temp_schema,table,5,0,0


In [33]:
sql = """
SELECT *
FROM incidents_money
LIMIT 5;
"""

# SELECT * , LIMIT 5 --- selects all columns but limits 5 rows
# from incidents_money relation

In [25]:
pd.read_sql(sql, conn)

,Incident_ID,Incident_Type,Estimated_Contents_Loss,Civilian_Injuries
0,2016050936,"EMS call, excluding vehicle accident with injury",0.0,NaN
1,2013022494,"EMS call, excluding vehicle accident with injury",0.0,NaN
2,2016048081,"Alarm system activation, no fire - unintentional",0.0,NaN
3,2021043224,Passenger vehicle fire,0.0,1.0
4,2018003513,"Excessive heat, scorch burns with no ignition",0.0,NaN


In [26]:
sql = """
SELECT *
FROM Category
LIMIT 5;
"""

In [27]:
pd.read_sql(sql, conn)

,Incident_Type,Category
0,"EMS call, excluding vehicle accident with injury",EMS
1,"Alarm system activation, no fire - unintentional",Alarm
2,Passenger vehicle fire,Fire
3,"Excessive heat, scorch burns with no ignition",Fire
4,Public service,Other


In [36]:
sql = """
SELECT DISTINCT Incident_Type 
FROM Category;"""

pd.read_sql(sql, conn)

,Incident_Type
0,"Alarm system activation, no fire - unintentional"
1,Alarm system sounded due to malfunction
2,Assist invalid
3,"Attempted burning, illegal action, other"
4,Building fire
5,CO detector activation due to malfunction
6,"Cooking fire, confined to container"
7,"Cover assignment, standby, moveup"
8,Dispatched & canceled en route
9,"EMS call, excluding vehicle accident with injury"


First, we are going to try to insert another row into the Incidents_Money table:

In [39]:
sql = """
INSERT INTO Incidents_Money
VALUES('201652372', 'Motorcycle Accident', 0, NULL );
"""

In [40]:
curs.execute(sql)

IntegrityError: FOREIGN KEY constraint failed

<span style="color:lightcyan">Why does this fail?</span>

The foreign key constraint was triggered because <span style="color:silver">"Motorcycle Accident" does not exist in the Category Table.</span> The databases refuses an incident type with no corresponding dictionary entry.

<span style="color:lightcyan">We then try to add this row (tuple) and it fails for a different reason...why?</span>

<span style="color:silver">Primary key value Incident Type is NULL,</span> which violates the NOT NULL constraint placed.

In [41]:
sql = """
INSERT INTO Incidents_Money
VALUES(NULL,'Building Fire',0,NULL);
"""

In [42]:
curs.execute(sql)

IntegrityError: NOT NULL constraint failed: incidents_money.Incident_ID

<span style="color:lightcyan">Next we would like to identify incidents where the contents loss is greater than $100 - write the query to do this.</span>

In [ ]:
sql = """
SELECT Incident_Type
FROM Incident_Money
WHERE Estimated_Contents_Loss > 100;
"""
# why choose Incident_ID, Civilian_Injuries? 
# I understand just chooseing Incident_ID, it uniquely identifies everything
# I dont' understand why my answer is wrong

In [ ]:
sql = """
SELECT Incident_ID, Civilian_Injuries
FROM Incidents_Money
WHERE Estimated_Contents_Loss > 100;
"""

In [ ]:
pd.read_sql(sql, conn)

,Incident_ID,Incident_Type,Estimated_Contents_Loss,Civilian_Injuries
0,2024041303,Passenger vehicle fire,500.0,NaN
1,2016022268,Building fire,3000.0,NaN
2,2018004712,Building fire,25000.0,1.0
3,2024015216,Building fire,15558.0,NaN


What is the relational algebra for this?

We would like to find Incidents where the Estimated Contents Loss is greater than the average amount. Write the query to do this.

In [ ]:
sql = """
SELECT Incident_ID, Civilian_Injuries
FROM Incidents_Money
WHERE Estimated_Contents_Loss >(
  SELECT AVG(Estimated_Contents_Loss)
  FROM Incidents_Money);
"""

In [ ]:
pd.read_sql(sql, conn)

,Incident_ID,Civilian_Injuries
0,2024041303,NaN
1,2016022268,NaN
2,2018004712,1.0
3,2024015216,NaN


We would like to find Incident_ID and Incident_Type for individual incidents of type Fire. Write the query to do this.

In [ ]:
sql = """
SELECT Incident_ID, im.Incident_Type
FROM Incidents_Money AS im
JOIN Category AS c
ON im.Incident_Type = c.Incident_Type
WHERE Category = "Fire";
"""

In [ ]:
sql = """
SELECT Incident_ID, Incident_Type
FROM Incidents_Money
JOIN Category
USING(Incident_Type)
WHERE Category = "Fire";
"""

In [ ]:
sql = """
SELECT Incident_ID, Incident_Type
FROM Incidents_Money
NATURAL JOIN Category
WHERE Category = "Fire";
"""

In [ ]:
pd.read_sql(sql, conn)

,Incident_ID,Incident_Type
0,2021043224,Passenger vehicle fire
1,2018003513,"Excessive heat, scorch burns with no ignition"
2,2022002096,Unauthorized burning
3,2021019118,Unauthorized burning
4,2018017710,"Cooking fire, confined to container"
5,2018014949,"Attempted burning, illegal action, other"
6,2024041303,Passenger vehicle fire
7,2016022268,Building fire
8,2022044406,"Heat from short circuit (wiring), defective/worn"
9,2018004712,Building fire


In [ ]:
pd.read_sql("SELECT * FROM Category LIMIT 2;", conn)

,Incident_Type,Category
0,"EMS call, excluding vehicle accident with injury",EMS
1,"Alarm system activation, no fire - unintentional",Alarm


What is a second way you could write this query?

In [ ]:
sql = """

"""

In [ ]:
pd.read_sql(sql, conn)

We are going to use a Union to get Incident_ID, Incident_Type and Category for all incidents in the category Fire and all incidents with >$100 loss. Then, we will do this a better way :)

In [ ]:
#first do this with a union of two subqueries
sql = """SELECT Incident_ID, Incident_Type, Category
FROM Incidents_Money
JOIN Category
USING(Incident_Type)
WHERE Category = 'Fire'
UNION
SELECT Incident_ID, Incident_Type, Category
FROM Incidents_Money
JOIN Category
USING(Incident_Type)
WHERE Estimated_Contents_Loss > 100"""

In [ ]:
pd.read_sql(sql, conn)

,Incident_ID,Incident_Type,Category
0,2011031066,Overheated motor,Fire
1,2015028988,Smoke or odor removal,Fire
2,2016020655,"Outside rubbish, trash or waste fire",Fire
3,2016022268,Building fire,Fire
4,2016048623,Passenger vehicle fire,Fire
5,2018003513,"Excessive heat, scorch burns with no ignition",Fire
6,2018004712,Building fire,Fire
7,2018014949,"Attempted burning, illegal action, other",Fire
8,2018017710,"Cooking fire, confined to container",Fire
9,2021019118,Unauthorized burning,Fire


In [ ]:
#first do this with a union of two subqueries
sql = """SELECT Incident_ID, Incident_Type, Category
FROM Incidents_Money
JOIN Category
USING(Incident_Type)
WHERE Category = 'Fire'
UNION ALL
SELECT Incident_ID, Incident_Type, Category
FROM Incidents_Money
JOIN Category
USING(Incident_Type)
WHERE Estimated_Contents_Loss > 100"""

In [ ]:
pd.read_sql(sql, conn)

,Incident_ID,Incident_Type,Category
0,2011031066,Overheated motor,Fire
1,2015028988,Smoke or odor removal,Fire
2,2016020655,"Outside rubbish, trash or waste fire",Fire
3,2016022268,Building fire,Fire
4,2016048623,Passenger vehicle fire,Fire
5,2018003513,"Excessive heat, scorch burns with no ignition",Fire
6,2018004712,Building fire,Fire
7,2018014949,"Attempted burning, illegal action, other",Fire
8,2018017710,"Cooking fire, confined to container",Fire
9,2021019118,Unauthorized burning,Fire


What is different between these two queries in terms of the output? Why?

What is a much simpler way to do this?

In [ ]:
sql = """
SELECT Incident_ID, Incident_Type, Category
FROM Incidents_Money
JOIN Category
USING(Incident_Type)
WHERE Category = 'Fire' OR Estimated_Contents_Loss > 100;
"""

In [ ]:
pd.read_sql(sql, conn)

,Incident_ID,Incident_Type,Category
0,2021043224,Passenger vehicle fire,Fire
1,2018003513,"Excessive heat, scorch burns with no ignition",Fire
2,2022002096,Unauthorized burning,Fire
3,2021019118,Unauthorized burning,Fire
4,2018017710,"Cooking fire, confined to container",Fire
5,2018014949,"Attempted burning, illegal action, other",Fire
6,2024041303,Passenger vehicle fire,Fire
7,2016022268,Building fire,Fire
8,2022044406,"Heat from short circuit (wiring), defective/worn",Fire
9,2018004712,Building fire,Fire


In [ ]:
conn.close()